In [2]:
# Import pandas for data manipulation and analysis
import pandas as pd

# Import NumPy for numerical operations and working with arrays
import numpy as np

# Import random for generating random values
import random

# Import datetime and timedelta for handling dates and time differences
from datetime import datetime, timedelta

# Import os for interacting with the operating system and file paths
import os

In [4]:
# Set a fixed random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Define the number of rows to generate in the synthetic dataset
NUM_ROWS = 15000

# Set the date range for generating sample sales records
START_DATE = datetime(2022, 1, 1)
END_DATE = datetime(2024, 12, 31)

# Define the file path where the generated dataset will be saved
OUTPUT_PATH = "../data/raw/novabay_sales_data.csv"

In [5]:
# Define Indian regions and their corresponding states for location-based analysis
regions_states = {
    "North": ["Delhi", "Punjab", "Haryana", "Uttar Pradesh"],
    "South": ["Karnataka", "Tamil Nadu", "Telangana", "Kerala"],
    "East":  ["West Bengal", "Odisha", "Bihar", "Jharkhand"],
    "West":  ["Maharashtra", "Gujarat", "Rajasthan", "Goa"]
}

# List of available sales channels used in the dataset
channels = ["Retail", "Wholesale", "Direct"]

# Customer segment categories for grouping buyers by business type
segments = ["SME", "Enterprise", "Distributor"]

# Product categories and the products included in each category
products = {
    "Food":          ["NovaBay Atta 5kg",    "NovaBay Rice 10kg",   "NovaBay Dal 1kg",    "NovaBay Oil 1L"],
    "Beverage":      ["NovaBay Juice 1L",    "NovaBay Water 500ml", "NovaBay Tea 250g",   "NovaBay Coffee 200g"],
    "Personal Care": ["NovaBay Shampoo 200ml","NovaBay Soap 100g",  "NovaBay Lotion 150ml","NovaBay Facewash 100ml"],
    "Home Care":     ["NovaBay Detergent 1kg","NovaBay Cleaner 500ml","NovaBay Dishwash 500ml","NovaBay Freshener 200ml"],
    "Snacks":        ["NovaBay Chips 100g",  "NovaBay Biscuits 200g","NovaBay Namkeen 150g","NovaBay Popcorn 100g"]
}

# Master list of 50 unique customers with customer IDs, names, and segment labels
customer_data = [
    ("C001", "Sharma Enterprises",    "SME"),
    ("C002", "Patel Distributors",    "Distributor"),
    ("C003", "Mehta Retail Chain",    "Enterprise"),
    ("C004", "Singh Wholesale Hub",   "Distributor"),
    ("C005", "Gupta General Store",   "SME"),
    ("C006", "Reddy Supermart",       "Enterprise"),
    ("C007", "Nair Trading Co",       "SME"),
    ("C008", "Joshi Brothers",        "Distributor"),
    ("C009", "Verma Retail Ltd",      "Enterprise"),
    ("C010", "Iyer Wholesale",        "Distributor"),
    ("C011", "Kapoor Traders",        "SME"),
    ("C012", "Malhotra Stores",       "Enterprise"),
    ("C013", "Chopra Distributors",   "Distributor"),
    ("C014", "Bose Retail Hub",       "SME"),
    ("C015", "Das Enterprises",       "Enterprise"),
    ("C016", "Pillai Trading",        "SME"),
    ("C017", "Rao Wholesale",         "Distributor"),
    ("C018", "Khan General Stores",   "SME"),
    ("C019", "Srivastava Retail",     "Enterprise"),
    ("C020", "Pandey Distributors",   "Distributor"),
    ("C021", "Mishra Traders",        "SME"),
    ("C022", "Tiwari Wholesale",      "Distributor"),
    ("C023", "Bajaj Supermart",       "Enterprise"),
    ("C024", "Agarwal Stores",        "SME"),
    ("C025", "Shah Distributors",     "Distributor"),
    ("C026", "Desai Retail Co",       "Enterprise"),
    ("C027", "Kulkarni Trading",      "SME"),
    ("C028", "Patil Wholesale",       "Distributor"),
    ("C029", "Jain Enterprises",      "Enterprise"),
    ("C030", "Thakur General Store",  "SME"),
    ("C031", "Bansal Distributors",   "Distributor"),
    ("C032", "Saxena Retail",         "Enterprise"),
    ("C033", "Rastogi Traders",       "SME"),
    ("C034", "Dubey Wholesale",       "Distributor"),
    ("C035", "Chaudhary Stores",      "Enterprise"),
    ("C036", "Tripathi Trading",      "SME"),
    ("C037", "Yadav Distributors",    "Distributor"),
    ("C038", "Sinha Retail Hub",      "Enterprise"),
    ("C039", "Kumar Wholesale",       "Distributor"),
    ("C040", "Shukla Enterprises",    "SME"),
    ("C041", "Naidu Traders",         "Enterprise"),
    ("C042", "Menon Wholesale",       "Distributor"),
    ("C043", "Krishnan Retail",       "SME"),
    ("C044", "Venkat Distributors",   "Distributor"),
    ("C045", "Murthy Stores",         "Enterprise"),
    ("C046", "Hegde Trading Co",      "SME"),
    ("C047", "Bhatt Wholesale",       "Distributor"),
    ("C048", "Pawar Retail Chain",    "Enterprise"),
    ("C049", "Gaikwad Traders",       "SME"),
    ("C050", "Deshpande Enterprises", "Distributor"),
]

In [6]:
# Assign order likelihood by region; West is intentionally weighted highest to drive more revenue
region_weights = {
    "North": 0.25,
    "South": 0.25,
    "East":  0.20,
    "West":  0.30   # Highest order share, so West contributes more to revenue
}

# Assign order likelihood by sales channel
channel_weights = {
    "Retail":    0.40,
    "Wholesale": 0.40,
    "Direct":    0.20
}

# Assign order likelihood by product category
category_weights = {
    "Food":          0.30,
    "Beverage":      0.25,
    "Personal Care": 0.20,
    "Home Care":     0.15,
    "Snacks":        0.10
}

# Define selling price ranges for each product category
price_ranges = {
    "Food":          (80,  400),
    "Beverage":      (30,  200),
    "Personal Care": (50,  300),
    "Home Care":     (60,  250),
    "Snacks":        (20,  120)
}

# Define cost-to-sales ratios for each category; Personal Care has the highest cost burden
cogs_ratio = {
    "Food":          (0.55, 0.65),
    "Beverage":      (0.55, 0.65),
    "Personal Care": (0.72, 0.80),  # Higher cost ratio, so margins are lower
    "Home Care":     (0.55, 0.65),
    "Snacks":        (0.55, 0.65)
}

# Define freight cost ranges by category; Personal Care has higher logistics cost
freight_ranges = {
    "Food":          (10, 40),
    "Beverage":      (10, 40),
    "Personal Care": (30, 80),   # Higher freight cost, which further reduces profit
    "Home Care":     (10, 40),
    "Snacks":        (5,  25)
}

# Define discount ranges by region; West receives the highest discounts to reduce margin
discount_ranges = {
    "North": (0.00, 0.25),
    "South": (0.00, 0.25),
    "East":  (0.00, 0.20),
    "West":  (0.10, 0.38)   # Higher discounting in West creates weaker profitability
}

# Define discount ranges by sales channel
channel_discount = {
    "Retail":    (0.00, 0.20),
    "Wholesale": (0.10, 0.35),  # Wholesale gets larger discounts
    "Direct":    (0.00, 0.15)    # Direct sales get the lowest discounts
}

# Assign customer-level order weights so some customers contribute more sales than others
# C003, C006, and C009 are intentionally made top revenue customers, but not top profit customers
customer_weights = [
    0.06, 0.04, 0.06, 0.04, 0.02,   # C001-C005
    0.06, 0.02, 0.04, 0.06, 0.04,   # C006-C010
    0.02, 0.04, 0.04, 0.02, 0.04,   # C011-C015
    0.02, 0.04, 0.02, 0.04, 0.04,   # C016-C020
    0.02, 0.02, 0.02, 0.02, 0.02,   # C021-C025
    0.02, 0.02, 0.02, 0.02, 0.02,   # C026-C030
    0.01, 0.01, 0.01, 0.01, 0.01,   # C031-C035
    0.01, 0.01, 0.01, 0.01, 0.01,   # C036-C040
    0.01, 0.01, 0.01, 0.01, 0.01,   # C041-C045
    0.01, 0.01, 0.01, 0.01, 0.01,   # C046-C050
]

# Apply extra discount pressure to the highest-revenue customers to intentionally reduce profit
high_revenue_customers = ["C003", "C006", "C009"]

In [9]:
rows = []

for i in range(NUM_ROWS):

    # --- Pick Order Date ---
    # Pick a random date within the configured date range
    days_range = (END_DATE - START_DATE).days
    order_date = START_DATE + timedelta(days=random.randint(0, days_range))

    # --- Pick Region and State ---
    # Select region using configured weights (probs sum to 1)
    region = np.random.choice(
        list(region_weights.keys()),
        p=list(region_weights.values())
    )
    # Pick a random state from the chosen region
    state = random.choice(regions_states[region])

    # --- Pick Channel ---
    # Select sales channel proportionally to its weight
    channel = np.random.choice(
        list(channel_weights.keys()),
        p=list(channel_weights.values())
    )

    # --- Pick Customer ---
    # Select a customer considering their order share (higher weight = more orders)
    customer = random.choices(customer_data, weights=customer_weights, k=1)[0]
    customer_id   = customer[0]        # e.g., C003
    customer_name = customer[1]        # e.g., "Mehta Retail Chain"
    customer_seg  = customer[2]        # e.g., "Enterprise"

    # --- Pick Category and Product ---
    # Select product category based on configured demand share
    category = np.random.choice(
        list(category_weights.keys()),
        p=list(category_weights.values())
    )
    # Pick a random item from the chosen category
    product_name = random.choice(products[category])
    # Auto-generate a compact product ID: P001_CAT
    product_id = "P" + str(products[category].index(product_name) + 1).zfill(3) + "_" + category[:3].upper()

    # --- Generate Order ID ---
    # Guarantee unique numeric order IDs
    order_id = "ORD" + str(100000 + i)

    # --- Units Sold ---
    # Randomly pick units per order
    units_sold = random.randint(1, 50)

    # --- Unit Price ---
    # Draw a price from the configured range for this category
    unit_price = round(random.uniform(*price_ranges[category]), 2)

    # --- Discount ---
    # Get discount range parameters for current region and channel
    region_disc_min,  region_disc_max  = discount_ranges[region]
    channel_disc_min, channel_disc_max = channel_discount[channel]

    # Use the higher of the two ranges to avoid being too aggressive in one dimension
    disc_min = max(region_disc_min, channel_disc_min)
    disc_max = max(region_disc_max, channel_disc_max)

    # Randomly pick a discount percentage within the effective range
    discount_pct = round(random.uniform(disc_min, disc_max), 2)

    # --- Extra discount for high‑revenue customers ---
    # Push even higher discounts to top‑revenue customers to weaken their profitability
    if customer_id in high_revenue_customers:
        discount_pct = round(min(discount_pct + random.uniform(0.08, 0.15), 0.40), 2)

    # --- COGS ---
    # Draw cost‑of‑goods ratio for the category and convert to per‑unit cost
    cogs_min, cogs_max = cogs_ratio[category]
    cogs_per_unit = round(random.uniform(cogs_min, cogs_max) * unit_price, 2)

    # --- Freight ---
    # Add freight cost drawn from category‑specific range
    freight_cost = round(random.uniform(*freight_ranges[category]), 2)
    
    # --- Revenue ---
    # Compute net revenue after applying discount at the order level
    revenue = round(unit_price * units_sold * (1 - discount_pct), 2)

    # --- Profit ---
    # Compute profit after subtracting cost of goods and freight for the order
    profit = round(revenue - (cogs_per_unit * units_sold) - freight_cost, 2)

    # --- Build Row ---
    # Bundle all fields into a dictionary row for the DataFrame
    rows.append({
        "order_id":         order_id,
        "order_date":       order_date.strftime("%Y-%m-%d"),
        "customer_id":      customer_id,
        "customer_name":    customer_name,
        "customer_segment": customer_seg,
        "region":           region,
        "state":            state,
        "sales_channel":    channel,
        "product_id":       product_id,
        "product_name":     product_name,
        "category":         category,
        "unit_price":       unit_price,
        "units_sold":       units_sold,
        "discount_pct":     discount_pct,
        "cogs_per_unit":    cogs_per_unit,
        "freight_cost":     freight_cost,
        "revenue":          revenue,
        "profit":           profit,
    })

In [11]:
# --- Convert to DataFrame ---
# Assemble the list of rows into a structured pandas DataFrame
df = pd.DataFrame(rows)

# --- Create output folder if it doesn't exist ---
# Ensure the output directory exists before saving
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# --- Save to CSV ---
# Write the synthetic sales dataset to CSV without index
df.to_csv(OUTPUT_PATH, index=False)

# -----------------------------------------------------------------------------
# --- Final output summary and model checks ---
# Provide a quick sanity‑check report on the generated dataset
# -----------------------------------------------------------------------------

print(f"Total rows generated: {len(df)}")
print(f"Date range: {df['order_date'].min()} to {df['order_date'].max()}")
print(f"Unique customers: {df['customer_id'].nunique()}")
print(f"Unique products: {df['product_id'].nunique()}")
print(f"Unique regions: {df['region'].nunique()}")
print(f"Categories: {df['category'].unique()}")

# Discount and margin diagnostics
print(f"Avg discount: {df['discount_pct'].mean():.2%}")
print(f"Avg profit margin: {(df['profit'].sum() / df['revenue'].sum()):.2%}")

print(f"\nRevenue by region:")
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

print(f"\nProfit margin by region:")
print((df.groupby('region')['profit'].sum() / df.groupby('region')['revenue'].sum()).sort_values())

print(f"\nProfit margin by category:")
print((df.groupby('category')['profit'].sum() / df.groupby('category')['revenue'].sum()).sort_values())

print(f"\nFile saved to: {OUTPUT_PATH}")

Total rows generated: 15000
Date range: 2022-01-01 to 2024-12-31
Unique customers: 50
Unique products: 20
Unique regions: 4
Categories: ['Snacks' 'Personal Care' 'Food' 'Home Care' 'Beverage']
Avg discount: 20.19%
Avg profit margin: 19.70%

Revenue by region:
region
West     14774363.18
South    12834744.87
North    12727630.66
East     10266434.50
Name: revenue, dtype: float64

Profit margin by region:
region
West     0.138312
North    0.216104
South    0.218899
East     0.230342
dtype: float64

Profit margin by category:
category
Personal Care    0.027871
Beverage         0.239142
Home Care        0.239732
Snacks           0.243485
Food             0.245228
dtype: float64

File saved to: ../data/raw/novabay_sales_data.csv


In [14]:
from IPython.display import display

# --- Convert to DataFrame ---
df = pd.DataFrame(rows)

# --- Create output folder if it doesn't exist ---
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# --- Save to CSV ---
df.to_csv(OUTPUT_PATH, index=False)

# --- Final Report ---
total_rows = len(df)
date_min = df['order_date'].min()
date_max = df['order_date'].max()
unique_customers = df['customer_id'].nunique()
unique_products = df['product_id'].nunique()
unique_regions = df['region'].nunique()
categories = ", ".join(df['category'].unique())
avg_discount = df['discount_pct'].mean()
avg_profit_margin = df['profit'].sum() / df['revenue'].sum()

revenue_by_region = df.groupby('region')['revenue'].sum().sort_values(ascending=False)
profit_margin_by_region = (df.groupby('region')['profit'].sum() / df.groupby('region')['revenue'].sum()).sort_values()
profit_margin_by_category = (df.groupby('category')['profit'].sum() / df.groupby('category')['revenue'].sum()).sort_values()

print("\n" + "=" * 70)
print(" NOVABAY SYNTHETIC SALES DATASET GENERATED SUCCESSFULLY ")
print("=" * 70)

print(f"\n📦 Rows Generated     : {total_rows:,}")
print(f"📅 Date Range         : {date_min} to {date_max}")
print(f"👥 Unique Customers   : {unique_customers}")
print(f"🛒 Unique Products    : {unique_products}")
print(f"🌍 Unique Regions     : {unique_regions}")
print(f"🏷️ Categories         : {categories}")
print(f"💸 Avg Discount       : {avg_discount:.2%}")
print(f"📈 Avg Profit Margin  : {avg_profit_margin:.2%}")

print("\n" + "-" * 70)
print(" REVENUE BY REGION ")
print("-" * 70)
display(revenue_by_region.to_frame(name="Revenue").style.format("{:,.2f}"))

print("\n" + "-" * 70)
print(" PROFIT MARGIN BY REGION ")
print("-" * 70)
display(profit_margin_by_region.to_frame(name="Profit Margin").style.format("{:.2%}"))

print("\n" + "-" * 70)
print(" PROFIT MARGIN BY CATEGORY ")
print("-" * 70)
display(profit_margin_by_category.to_frame(name="Profit Margin").style.format("{:.2%}"))

print("\n" + "=" * 70)
print(f" File saved to: {OUTPUT_PATH}")
print("=" * 70)


 NOVABAY SYNTHETIC SALES DATASET GENERATED SUCCESSFULLY 

📦 Rows Generated     : 15,000
📅 Date Range         : 2022-01-01 to 2024-12-31
👥 Unique Customers   : 50
🛒 Unique Products    : 20
🌍 Unique Regions     : 4
🏷️ Categories         : Snacks, Personal Care, Food, Home Care, Beverage
💸 Avg Discount       : 20.19%
📈 Avg Profit Margin  : 19.70%

----------------------------------------------------------------------
 REVENUE BY REGION 
----------------------------------------------------------------------


,Revenue
region,
West,"14,774,363.18"
South,"12,834,744.87"
North,"12,727,630.66"
East,"10,266,434.50"



----------------------------------------------------------------------
 PROFIT MARGIN BY REGION 
----------------------------------------------------------------------


,Profit Margin
region,
West,13.83%
North,21.61%
South,21.89%
East,23.03%



----------------------------------------------------------------------
 PROFIT MARGIN BY CATEGORY 
----------------------------------------------------------------------


,Profit Margin
category,
Personal Care,2.79%
Beverage,23.91%
Home Care,23.97%
Snacks,24.35%
Food,24.52%



 File saved to: ../data/raw/novabay_sales_data.csv
